In [ ]:
import arcpy
import os

In [ ]:
#This code produces a dbf for each raster

#Counties

#Contiguous usa rasters
project = arcpy.mp.ArcGISProject("CURRENT")
gdb = project.defaultGeodatabase
project_folder = os.path.dirname(project.filePath)


counties = os.path.join(gdb, "FarmGO_Counties")

raster_folder = (r"")

arcpy.CheckOutExtension("Spatial")

# ------------------------------------------------------------
# OUTPUT FOLDER
# ------------------------------------------------------------

zonal_folder = os.path.join(
    project_folder,
    "ZonalStatsTablesUpdated"
)

os.makedirs(zonal_folder, exist_ok=True)

print(f"Zonal statistics will be saved to:")
print(zonal_folder)

# ------------------------------------------------------------
# SPATIAL ANALYST
# ------------------------------------------------------------

arcpy.CheckOutExtension("Spatial")

# ------------------------------------------------------------
# PROCESS ALL PRISM RASTERS
# ------------------------------------------------------------

for root, dirs, files in os.walk(raster_folder):

    for file in files:

        # Only process TIFF rasters
        if not file.lower().endswith((".tif", ".tiff")):
            continue

        raster_path = os.path.join(root, file)

        # ----------------------------------------------------
        # GET VARIABLE NAME
        # ----------------------------------------------------

        # Example:
        # ...\PRISM_5070\tmin\prism_tmin_us_30s_202608.tif

        variable = os.path.basename(root)

        # Remove raster extension
        raster_name = os.path.splitext(file)[0]

        # ----------------------------------------------------
        # OUTPUT TABLE NAME
        # ----------------------------------------------------

        table_name = f"Zonal_{variable}_{raster_name}.dbf"

        output_table = os.path.join(
            zonal_folder,
            table_name
        )

        print("----------------------------------------")
        print(f"Variable: historical_{variable}")
        print(f"Raster:   {file}")
        print(f"Output:   {table_name}")

        # ----------------------------------------------------
        # ZONAL STATISTICS
        # ----------------------------------------------------

        arcpy.sa.ZonalStatisticsAsTable(
            in_zone_data=counties,
            zone_field="GEOID",
            in_value_raster=raster_path,
            out_table=output_table,
            ignore_nodata="DATA",
            statistics_type="MEAN"
        )

print("----------------------------------------")
print("Finished all PRISM zonal statistics.")
print(f"Tables saved to: {zonal_folder}")

arcpy.CheckInExtension("Spatial")

In [ ]:
#This code samples rasters at field centroids 

#Fields

# ============================================================
# PROJECT SETUP
# ============================================================

project = arcpy.mp.ArcGISProject("CURRENT")
gdb = project.defaultGeodatabase
project_folder = os.path.dirname(project.filePath)

fields = os.path.join(
    gdb,
    "TexasFields_FTW_2025"
)

raster_folder = (
    r"E:\JOB_STUFF\Normals\Monthlies\PRISM_Unzipped\PRISM_5070\monthly")

arcpy.CheckOutExtension("Spatial")


# ============================================================
# OUTPUT FOLDER
# ============================================================

zonal_folder = os.path.join(
    project_folder,
    "ZonalStatsTablesFieldsNewUpdated"
)

os.makedirs(
    zonal_folder,
    exist_ok=True
)

print("Raster values will be saved to:")
print(zonal_folder)


# ============================================================
# TEMPORARY FIELD CENTROIDS
# ============================================================

centroids = os.path.join(
    gdb,
    "FieldCentroids"
)

if arcpy.Exists(centroids):

    print()
    print("FieldCentroids already exists.")
    print("Using existing centroid layer.")

else:

    print()
    print("Creating field centroid points...")
    print("This may take some time for 2+ million fields.")

    arcpy.management.FeatureToPoint(
        in_features=fields,
        out_feature_class=centroids,
        point_location="INSIDE"
    )

    print("Field centroids created.")


# ============================================================
# VERIFY FIELD_ID
# ============================================================

centroid_fields = {
    field.name.upper(): field.name
    for field in arcpy.ListFields(centroids)
}

if "FIELD_ID" not in centroid_fields:

    raise ValueError(
        "FieldCentroids does not contain FIELD_ID."
    )


field_id_field = centroid_fields["FIELD_ID"]


# ============================================================
# PROCESS ALL PRISM RASTERS
# ============================================================

for root, dirs, files in os.walk(raster_folder):

    for file in files:

        # ----------------------------------------------------
        # ONLY PROCESS TIFF RASTERS
        # ----------------------------------------------------

        if not file.lower().endswith(
            (".tif", ".tiff")
        ):
            continue


        raster_path = os.path.join(
            root,
            file
        )


        # ----------------------------------------------------
        # GET VARIABLE NAME
        # ----------------------------------------------------

        # Example:
        #
        # PRISM_5070
        # └── tmin
        #     └── prism_tmin_us_30s_202608.tif

        variable = os.path.basename(root)


        # Remove raster extension
        raster_name = os.path.splitext(file)[0]


        # ----------------------------------------------------
        # OUTPUT TABLE NAME
        # ----------------------------------------------------

        table_name = (
            f"Zonal_{variable}_{raster_name}.dbf"
        )

        output_table = os.path.join(
            zonal_folder,
            table_name
        )


        print()
        print("----------------------------------------")
        print(f"Variable: {variable}")
        print(f"Raster:   {file}")
        print(f"Output:   {table_name}")


        # ----------------------------------------------------
        # SKIP IF ALREADY EXISTS
        # ----------------------------------------------------

        if arcpy.Exists(output_table):

            print(
                "Output already exists - skipping."
            )

            continue


        # ----------------------------------------------------
        # CREATE TEMPORARY RASTER VALUE FIELD NAME
        # ----------------------------------------------------

        raster_value_field = "PRISM_VAL"


        # ----------------------------------------------------
        # COPY CENTROIDS
        #
        # ExtractMultiValuesToPoints modifies the point
        # feature class by adding a field, so we make a
        # temporary copy for each raster.
        # ----------------------------------------------------

        temp_points = os.path.join(
            gdb,
            "TempPRISMPoints"
        )


        if arcpy.Exists(temp_points):

            arcpy.management.Delete(
                temp_points
            )


        arcpy.management.CopyFeatures(
            centroids,
            temp_points
        )


        # ----------------------------------------------------
        # EXTRACT RASTER VALUE
        # ----------------------------------------------------
        
        arcpy.sa.ExtractMultiValuesToPoints(
            temp_points,
            [[raster_path, raster_value_field]],
            "NONE"
        )
        
        print("Raster values extracted.")


        # ----------------------------------------------------
        # CREATE OUTPUT TABLE
        # ----------------------------------------------------

        arcpy.management.CreateTable(
            zonal_folder,
            table_name
        )


        arcpy.management.AddField(
            output_table,
            "FIELD_ID",
            "TEXT",
            field_length=30
        )


        arcpy.management.AddField(
            output_table,
            "MEAN",
            "DOUBLE"
        )


        # ----------------------------------------------------
        # COPY FIELD_ID + RASTER VALUE
        # ----------------------------------------------------

        inserted = 0


        with arcpy.da.SearchCursor(
            temp_points,
            [
                field_id_field,
                raster_value_field
            ]
        ) as search_cursor:

            with arcpy.da.InsertCursor(
                output_table,
                [
                    "FIELD_ID",
                    "MEAN"
                ]
            ) as insert_cursor:

                for field_id, value in search_cursor:

                    # Keep the row even if the raster value
                    # is NoData.
                    #
                    # This allows us to distinguish:
                    # field exists but no raster value
                    #
                    # from:
                    # field doesn't exist in the table.

                    insert_cursor.insertRow(
                        (
                            str(field_id)
                            if field_id is not None
                            else None,
                            value
                        )
                    )

                    inserted += 1


        print(
            f"Added {inserted:,} field records."
        )


        # ----------------------------------------------------
        # DELETE TEMPORARY POINTS
        # ----------------------------------------------------

        arcpy.management.Delete(
            temp_points
        )


# ============================================================
# COMPLETE
# ============================================================

print()
print("----------------------------------------")
print("Finished all PRISM field extraction.")
print()
print(f"Tables saved to:")
print(zonal_folder)

arcpy.CheckInExtension("Spatial")